# HEAT1: does short-term heat transfer between correlated stocks?

Deep models (BiLSTM, CNN-ResNet, Transformer) read the last hour of a stock and of its 5 most
correlated peers, and predict its market-relative return over the next 30 minutes, 1, 2 and 4 hours.
Two controls, same model and budget: **no peers** (the stock's own reversal only) and **5 random
peers** (E12's control). Transfer = how much the peers cut the prediction error, in nats. The
**heat map** says which stock's moves feed which, and the **visual report** draws all of it.

Data: the thesis 5-minute SIP bars on Drive (S&P 500, 2016–2026). The rule is registered in
[`configs/heat_v1.yaml`](https://github.com/Gaire-commits/Transfer-entropy-/blob/main/configs/heat_v1.yaml)
(2026-10-20). **Edge** = all of: net of 2 bps above zero on the test days (2024 onward, 99% day-block
interval), beats no peers, beats random peers, positive in both halves, survives false-discovery
control over the 12 cells.

**The order**
1. Setup, then controls.
2. **Check run** (2021 only, never the test days): proves the pipeline works on the real data.
3. **Tune** the deep models on the *validation* days (2022–2023). Never the test days.
4. **Register** the tuned settings as `heat_v2` (before any test scoring), then run it.
5. Result and the visual report.

Tuning is optional: you can skip section 4 and run section 5 with the untuned `heat_v1`.

**Before anything else, pick the GPU:** Runtime → Change runtime type → T4 or better (High-RAM if
offered). Changing the runtime type later resets the session, and then Setup must be run again.
Run top to bottom. Don't use "Run all" once you reach section 5: it scores the test days.

The big panel file stays on this machine's disk (it rebuilds in a few minutes); only the small result
files go to Drive. Every cell that runs a script ends by flushing Drive (`Drive synced.`) with a time
limit, so a slow upload can never freeze the notebook. If the runtime dies *during* a script, anything it hadn't finished is lost; the
architectures already scored are safe, and re-running the cell picks up from them.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
    !pip install -q pyyaml
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO)
sys.path.insert(0, REPO)
for name in [m for m in sys.modules if m == "heat" or m.startswith("heat.")]:
    del sys.modules[name]  # re-running this cell after a pull uses the new code

import html as _html
import torch
from IPython.display import HTML, Markdown, display

DATA = os.environ.get("HEAT_DATA_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/data/bars_sip")
OUT = os.environ.get("HEAT_OUT_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1")
# The panel is ~420 MB and rebuilds in a few minutes, so it lives on this machine's disk, not Drive:
# one huge upload can hold up the small result files queued behind it. Only results go to Drive.
PANEL = os.environ.get("HEAT_PANEL", "/content/heat_panel.npz")


def sync_drive(timeout=180):
    """Colab uploads to Drive in the background and a runtime restart can lose what hasn't gone up, so flush.
    Never hangs: if the upload takes longer than `timeout` seconds it says so and moves on (the files keep
    uploading while this session lives). If Colab asks you to authorize again after the flush, click through."""
    if not IN_COLAB:
        return
    import threading

    failed = []

    def flush():
        try:
            drive.flush_and_unmount()
        except Exception as exc:  # noqa: BLE001
            failed.append(exc)
            print("(could not flush Drive:", exc, ")")

    t = threading.Thread(target=flush, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        print(f"Drive is still uploading after {timeout}s. Your results are on this machine and keep uploading "
              "in the background; ask Claude to check Drive again in a few minutes.")
        return
    try:
        drive.mount("/content/drive")
        print("Drive NOT confirmed synced." if failed else "Drive synced.")
    except Exception as exc:  # noqa: BLE001
        print("(Drive flushed but could not be remounted:", exc, "- run drive.mount('/content/drive') yourself)")


def download_results(out_dir, name):
    """Zip a run's results (reports, tables, the visual report; not the model weights) and download it
    to your computer. A copy that doesn't depend on Drive."""
    import zipfile
    out_dir = os.path.abspath(out_dir)
    if not os.path.isdir(out_dir):
        print("Nothing to download yet:", out_dir, "does not exist.")
        return
    path = f"/content/{name}.zip" if IN_COLAB else os.path.join(out_dir, f"{name}.zip")
    keep = (".md", ".csv", ".json", ".html", ".yaml")
    n = 0
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out_dir):
            for f in files:
                if f.endswith(keep):
                    z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), out_dir))
                    n += 1
    print(f"{n} files in {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(path)


def show_viz(out_dir, cfg_path="configs/heat_v1.yaml", height=1800):
    """Write <out_dir>/viz.html and show it here (it is also a file you can open full-screen)."""
    from heat import config
    from heat.viz import write_report
    cfg = config.load(cfg_path)
    meta = {"experiment": cfg["experiment"], "cost_bps": cfg["evaluation"]["cost_bps"],
            "level": cfg["evaluation"]["level"], "check": os.path.basename(out_dir) == "check"}
    path = write_report(out_dir, cfg["data"]["universe"], meta, PANEL)
    print("saved", path, "- download it from Drive to open it full-screen in any browser")
    page = open(path).read()
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # IPython suggests IFrame, which can't read a Drive file
        display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:1px solid #8885;border-radius:8px"></iframe>'
                     % (_html.escape(page, quote=True), height)))


if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    tpu = bool(os.environ.get("TPU_NAME") or os.environ.get("COLAB_TPU_ADDR") or os.path.exists("/dev/accel0")
               or os.path.exists("/dev/vfio"))
    print("=" * 78)
    print("NO GPU. " + ("This is a TPU runtime: this code only uses NVIDIA GPUs, so it would run on the CPU, "
                        "many times slower." if tpu else "Training would run on the CPU, many times slower."))
    print("Fix: Runtime -> Change runtime type -> T4 GPU (or L4 / A100), then run this Setup cell again.")
    print("=" * 78)
print("SIP bars found:", os.path.isdir(os.path.join(DATA, "5Min")))
print("outputs:", OUT)

## 2. Controls (about 5 minutes)

Before real data: on synthetic stocks with a planted leader, the experiment must find the edge and
name the leader in the heat map; with nothing planted it must find nothing. Plus the no-look-ahead
checks, the tuning checks, the visual-report checks and the frozen-config check.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} -m pytest -q tests -W ignore

## 3. Check run on the real data (inside the training years)

Builds the panel once (every stock's market-relative returns on the thesis grid; a few minutes,
saved to the outputs folder), then a short run that trains to 2019, validates on 2020 and scores
**2021**. It never touches the registered test days, so it's safe to run as often as needed. It shows
whether everything works and gives a feel for the numbers; it is not the result.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --data-dir "$DATA" --out-dir "$OUT" --check --panel "$PANEL"
    report = os.path.join(OUT, "check", "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report: the run above stopped with an error. Send a screenshot of its last lines.")
    sync_drive()  # make sure what the script wrote is really on Drive

### The check run, drawn

Transfer by hold, who leads whom (matrix and network), industries leading industries, and who moves
with whom. These are the 2021 numbers from the check run, not the result.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(OUT, "check", "cells.csv")) or os.path.exists(os.path.join(OUT, "check", "cells_partial.csv")):
        show_viz(os.path.join(OUT, "check"))
    else:
        print("No check results yet: run the check cell above first.")

**Keep a copy.** Downloads the check run's results to your computer as a zip (reports, tables, the
visual report). It doesn't depend on Drive, so do it even if Drive is slow. Send the zip, or a screenshot,
if Claude can't see the files on Drive.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(os.path.join(OUT, "check"), "heat_check_results")

## 4. Tune the deep models (validation days only)

Random search with successive halving over learning rate, weight decay, width, depth, dropout, and
each architecture's own knobs (kernel size, attention heads). It only ever sees the training days
(2016–2021) and the validation days (2022–2023); **the test days are never loaded**. The registered
default is always one candidate, and the three variants (no peers, correlated, random) share each
setting, so the peer models get no tuning advantage. The winner is adopted only if it beats the
default by more than the seed-to-seed spread; otherwise the default is kept.

Each architecture is saved to Drive when done and skipped if you run its cell again.

**4a. Quick check (optional, a few minutes):** two trials on 2019–2020, just to see the tuner runs.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --check --trials 3 --budgets 1 2 --archs resnet1d
    sync_drive()  # make sure what the script wrote is really on Drive

**4b. The real tuning**, one architecture per cell. It prints one line per trial, so you can watch it work.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --archs bilstm
    sync_drive()  # make sure what the script wrote is really on Drive

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --archs resnet1d
    sync_drive()  # make sure what the script wrote is really on Drive

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --archs transformer
    sync_drive()  # make sure what the script wrote is really on Drive

**4c. What it chose.** When all three are tuned, `tuning/heat_v2.yaml` is the candidate registered
config. **Do not run section 5 yet:** it has to be registered first (committed to the repo with its hash
frozen) so the rule is fixed before the test days are scored. Tell Claude the tuning is done; it reads
the files from Drive, registers `heat_v2`, and pushes. Then re-run Setup (it pulls) and continue.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    report = os.path.join(OUT, "tuning", "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
        cand = os.path.join(OUT, "tuning", "heat_v2.yaml")
        print("candidate config written:" , os.path.exists(cand), cand)
    else:
        print("No tuning report yet: run the tuning cells above.")

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(os.path.join(OUT, "tuning"), "heat_tuning_results")  # a copy, independent of Drive

## 5. The registered run: scores the test days

One architecture per cell, so each fits in a Colab session. Each is saved to Drive as soon as it's
scored, and a scored one is skipped if you run its cell again, so a dropped session just resumes. The
verdict appears once all three are in.

The first cell picks the registered config: `heat_v2` (tuned) if it has been registered in the repo,
otherwise `heat_v1`. It prints which one, so you always know the rule you're running.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    import yaml
    CONFIG = "configs/heat_v2.yaml" if os.path.exists("configs/heat_v2.yaml") else "configs/heat_v1.yaml"
    EXP = yaml.safe_load(open(CONFIG))["experiment"]
    REG_OUT = os.path.join(os.path.dirname(OUT), EXP)
    print("registered config:", CONFIG, "->", EXP)
    print("results go to:", REG_OUT)
    if EXP == "heat_v1":
        print("(untuned. If you tuned, tell Claude to register heat_v2, then re-run Setup and this cell.)")

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --archs bilstm
    sync_drive()  # make sure what the script wrote is really on Drive

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --archs resnet1d
    sync_drive()  # make sure what the script wrote is really on Drive

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --archs transformer
    sync_drive()  # make sure what the script wrote is really on Drive

## 6. Result

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    report = os.path.join(REG_OUT, "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report yet: run the cells in section 5 first.")

### The result, drawn

Use the filters at the top of the report: architecture (or all three averaged), hold, and how many
links to show. Hover any square, arrow or dot for the numbers. Every chart has a table view.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(REG_OUT, "cells.csv")) or os.path.exists(os.path.join(REG_OUT, "cells_partial.csv")):
        show_viz(REG_OUT, CONFIG)
    else:
        print("No results yet: run the cells in section 5 first.")

**Keep a copy** of the registered result on your computer (independent of Drive).

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(REG_OUT, EXP + "_results")